Codice per l'addestramento di un modello alla predizione di cwe a partire dalla descrizione delle cve, opzionalmente con la concatenazione di CVSS score e CPE nell'input

In [1]:
#librerie
!pip install transformers torch scikit-learn pandas numpy accelerate -q
import pandas as pd
import numpy as np
import torch
import os, json, joblib, pickle
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizzo: {device}")
print(f"GPU: {torch.cuda.is_available()}")

Utilizzo: cuda
GPU: True


In [ ]:
#lettura csv
df = pd.read_csv('datasetcompleto.csv')
print(df.head())
print(f"\nDimensione csv: {df.shape}")
print(f"\nNomi colonne csv: {df.columns.tolist()}")

CVE_COL = 'cve_id'
DESC_COL = 'description'
CWE_COL = 'cwe'
CVSS_COL = 'cvss_vector'
CPE_COL = 'cpe'

#rimozione righe in cui manca descrizione cve o la cwe
df = df.dropna(subset=[DESC_COL, CWE_COL])
print(f"\nDimensione dataset dopo rimozione: {df.shape}")


USE_CVSS = True   #include CVSS nell'input
USE_CPE  = True   #include CPE nell'input

def is_valid_field(x):
    if pd.isna(x):
        return False
    s = str(x).strip()
    return (s != "") and (s.upper() != "N/A")

def build_input_text(row, use_cvss=True, use_cpe=True):
    parts = []
    if use_cvss and is_valid_field(row.get(CVSS_COL, None)):
        parts.append(f"[CVSS: {str(row[CVSS_COL]).strip()}]")
    if use_cpe and is_valid_field(row.get(CPE_COL, None)):
        parts.append(f"[CPE: {str(row[CPE_COL]).strip()}]")
    parts.append(str(row[DESC_COL]).strip())
    return " ".join(parts)

df['combined_text'] = df.apply(build_input_text, axis=1, use_cvss=USE_CVSS, use_cpe=USE_CPE)

print("\nEsempio di testo concatenato")
print(df['combined_text'].iloc[0])

          cve_id                                        description  \
0  CVE-1999-0095  The debug command in Sendmail is enabled, allo...   
1  CVE-1999-0082      CWD ~root command in ftpd allows root access.   
2  CVE-1999-1471  Buffer overflow in passwd in BSD based operati...   
3  CVE-1999-1122  Vulnerability in restore in SunOS 4.0.3 and ea...   
4  CVE-1999-1467  Vulnerability in rcp on SunOS 4.0.x allows rem...   

             cwe  cvss_score                 cvss_vector cvss_version  \
0  NVD-CWE-Other        10.0  AV:N/AC:L/Au:N/C:C/I:C/A:C         v2.0   
1  NVD-CWE-Other        10.0  AV:N/AC:L/Au:N/C:C/I:C/A:C         v2.0   
2  NVD-CWE-Other         7.2  AV:L/AC:L/Au:N/C:C/I:C/A:C         v2.0   
3  NVD-CWE-Other         4.6  AV:L/AC:L/Au:N/C:P/I:P/A:P         v2.0   
4  NVD-CWE-Other        10.0  AV:N/AC:L/Au:N/C:C/I:C/A:C         v2.0   

                                                 cpe  
0  cpe:2.3:a:eric_allman:sendmail:5.58:*:*:*:*:*:*:*  
1  cpe:2.3:a:ftp:ftp:*:*

In [3]:
#parsing colonna cwe
def parse_cwe(cwe_string):
    if pd.isna(cwe_string) or cwe_string == '':
        return []
    cwes = [cwe.strip() for cwe in str(cwe_string).split(',')]
    cwes = [cwe.split()[0] if ' ' in cwe else cwe for cwe in cwes]
    return cwes

df['CWE_list'] = df[CWE_COL].apply(parse_cwe)

#rimozione cwe NVD-CWE-Other e NVD-CWE-noinfo
def has_valid_cwe(cwe_list):
    if not cwe_list:
        return False
    valid_cwes = [cwe for cwe in cwe_list
                  if cwe not in ['NVD-CWE-Other', 'NVD-CWE-noinfo']]
    return len(valid_cwes) > 0

df = df[df['CWE_list'].apply(has_valid_cwe)].copy()

print(f"numero righe dopo il filtro NVD-CWE-Other e NVD-CWE-noinfo {len(df)}")

def clean_cwe_list(cwe_list):
    return [cwe for cwe in cwe_list
            if cwe not in ['NVD-CWE-Other', 'NVD-CWE-noinfo']]
df['CWE_list'] = df['CWE_list'].apply(clean_cwe_list)

# MultiLabelBinarizer (ogni cwe diventa un valore di un vettore lungo il numero di label)
mlb = MultiLabelBinarizer()
cwe_encoded = mlb.fit_transform(df['CWE_list'])

num_labels = len(mlb.classes_)
print(f"\nnumero di label: {num_labels}")

numero righe dopo il filtro NVD-CWE-Other e NVD-CWE-noinfo 253282

numero di label: 754


In [ ]:
class CVEDataset(Dataset): #trasforma in dataset compatibile con il trainer
    def __init__(self, texts, labels, tokenizer, max_length=384):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(label)
        }

# split train/validation/test (70/15/15)
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['combined_text'].values,
    cwe_encoded,
    test_size=0.3,
    random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42
)


In [ ]:
# carica SecureBERT
model_name = "ehsanaghaei/SecureBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = CVEDataset(train_texts, train_labels, tokenizer)
val_dataset = CVEDataset(val_texts, val_labels, tokenizer)
test_dataset = CVEDataset(test_texts, test_labels, tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    trust_remote_code=True
)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at ehsanaghaei/SecureBERT and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
#metriche
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions
    preds_sigmoid = 1 / (1 + np.exp(-preds))
    preds_binary = (preds_sigmoid > 0.5).astype(int) 

    f1_micro = f1_score(labels, preds_binary, average='micro', zero_division=0)
    f1_macro = f1_score(labels, preds_binary, average='macro', zero_division=0)
    precision = precision_score(labels, preds_binary, average='micro', zero_division=0)
    recall = recall_score(labels, preds_binary, average='micro', zero_division=0)
    exact_match = np.all(labels == preds_binary, axis=1).mean() #PERCENTUALE DI CVE CON TUTTE LE CWE CORRETTE (ACCURACY)

    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'precision': precision,
        'recall': recall,
        'exact_match_ratio': exact_match
    }

In [7]:

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=7,  #EPOCHE
    per_device_train_batch_size=32,  #batch size
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model='f1_micro',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

# ricerca threshold ottimale
val_predictions = trainer.predict(val_dataset)
val_preds_sigmoid = 1 / (1 + np.exp(-val_predictions.predictions))

thresholds = np.arange(0.3, 0.7, 0.05)
best_threshold = 0.5
best_f1 = 0

print("\nottimizzazione threshold:")
for threshold in thresholds:
    preds_binary = (val_preds_sigmoid > threshold).astype(int)
    f1 = f1_score(val_labels, preds_binary, average='micro')
    print(f"Threshold {threshold:.2f}: F1 Micro = {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"\nmigliore threshold: {best_threshold:.2f} con F1 Micro: {best_f1:.4f}")

#validation set
val_results = trainer.evaluate()
print(f"\nmetriche su validation set:")
for key, value in val_results.items():
    print(f"{key}: {value:.4f}")

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Precision,Recall,Exact Match Ratio
1,0.007700,0.007611,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.006300,0.005953,0.334782,0.002559,0.956654,0.202892,0.205438
3,0.003300,0.003205,0.677418,0.034639,0.877647,0.551579,0.545694
4,0.002900,0.002862,0.705811,0.048752,0.863742,0.596706,0.586071
5,0.002600,0.002743,0.719532,0.056754,0.868458,0.614206,0.607628
6,0.002500,0.002696,0.725634,0.060724,0.862018,0.626511,0.617841
7,0.002500,0.002694,0.726958,0.061498,0.860383,0.629360,0.619999



ottimizzazione threshold:
Threshold 0.30: F1 Micro = 0.7303
Threshold 0.35: F1 Micro = 0.7339
Threshold 0.40: F1 Micro = 0.7340
Threshold 0.45: F1 Micro = 0.7311
Threshold 0.50: F1 Micro = 0.7270
Threshold 0.55: F1 Micro = 0.7210
Threshold 0.60: F1 Micro = 0.7145
Threshold 0.65: F1 Micro = 0.7077

migliore threshold: 0.40 con F1 Micro: 0.7340



metriche su validation set:
eval_loss: 0.0027
eval_f1_micro: 0.7270
eval_f1_macro: 0.0615
eval_precision: 0.8604
eval_recall: 0.6294
eval_exact_match_ratio: 0.6200
eval_runtime: 73.4694
eval_samples_per_second: 517.1140
eval_steps_per_second: 8.0850
epoch: 7.0000


In [9]:

# test set
test_results = trainer.evaluate(test_dataset)

for key, value in test_results.items():
    if key.startswith('eval_'):
        metric_name = key.replace('eval_', '').upper()
        print(f"{metric_name}: {value:.4f}")

predictions = trainer.predict(test_dataset)
preds_sigmoid = 1 / (1 + np.exp(-predictions.predictions))
preds_binary = (preds_sigmoid > best_threshold).astype(int)  # USA best_threshold

f1_micro = f1_score(test_labels, preds_binary, average='micro', zero_division=0)
f1_macro = f1_score(test_labels, preds_binary, average='macro', zero_division=0)
precision = precision_score(test_labels, preds_binary, average='micro', zero_division=0)
recall = recall_score(test_labels, preds_binary, average='micro', zero_division=0)
exact_match = np.all(test_labels == preds_binary, axis=1).mean()

print(f"\n=== RISULTATI TEST SET CON THRESHOLD OTTIMIZZATO ({best_threshold:.2f}) ===")
print(f"F1_MICRO: {f1_micro:.4f}")
print(f"F1_MACRO: {f1_macro:.4f}")
print(f"PRECISION: {precision:.4f}")
print(f"RECALL: {recall:.4f}")
print(f"EXACT_MATCH_RATIO: {exact_match:.4f}")

avg_labels_true = test_labels.sum(axis=1).mean()
avg_labels_pred = preds_binary.sum(axis=1).mean()

print(f"media CWE per CVE nel dataset: {avg_labels_true:.2f}")
print(f"media CWE per CVE predette: {avg_labels_pred:.2f}")

LOSS: 0.0027
F1_MICRO: 0.7320
F1_MACRO: 0.0619
PRECISION: 0.8665
RECALL: 0.6336
EXACT_MATCH_RATIO: 0.6253
RUNTIME: 74.0827
SAMPLES_PER_SECOND: 512.8460
STEPS_PER_SECOND: 8.0180

=== RISULTATI TEST SET CON THRESHOLD OTTIMIZZATO (0.40) ===
F1_MICRO: 0.7384
F1_MACRO: 0.0692
PRECISION: 0.8257
RECALL: 0.6678
EXACT_MATCH_RATIO: 0.6468
media CWE per CVE nel dataset: 1.10
media CWE per CVE predette: 0.89


In [10]:
save_dir = "./modellocwe"
os.makedirs(save_dir, exist_ok=True)

#salvataggio modello e tokenizer
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# salvataggio multilabelbinarizer
joblib.dump(mlb, os.path.join(save_dir, "mlb.joblib"))

# salvataggio threshold e max length
with open(os.path.join(save_dir, "extra_config.json"), "w") as f:
    json.dump({"best_threshold": float(best_threshold), "max_length": 384}, f, indent=2)

print("modello salvato in:", os.path.abspath(save_dir))

#salvataggio pkl

pkl_path = os.path.join(save_dir, "pklmodello.pkl")

bundle = {
    "model": trainer.model,
    "tokenizer": tokenizer,
    "mlb": mlb,
    "best_threshold": float(best_threshold),
    "max_length": 384
}

with open(pkl_path, "wb") as f:
    pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print("pickle salvato in:", pkl_path)

modello salvato in: /content/modellocwe
pickle salvato in: ./modellocwe/pklmodello.pkl
